# Practice: Decide Whether Your Task Needs Sequence

**Computational Text Analysis Pack · Module 24 — Sequence Models**

*Technique:* A decision experiment: measure the order signal before buying the model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/12-neural-foundations/24c_practice_does_order_help.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## The question

> **Before spending a week on a sequence model, how do you find out whether your
> task contains any order signal at all?**

There is a cheap test, and almost nobody runs it: destroy the word order and see
how much accuracy you lose. Whatever you lose is the ceiling on what a sequence
model can win back.

### How to use this notebook

A worked case, not a tutorial. Run it once for the shape of the argument, then
change the question at the top to one of your own.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

rng = np.random.default_rng(0)
sub = df[df["n_words"] >= 30].sample(n=4000, random_state=1).reset_index(drop=True)
def scramble(t):
    toks = tokenise(str(t)); rng.shuffle(toks); return " ".join(toks)
sub["scrambled"] = sub["text"].map(scramble)
tr, te = train_test_split(np.arange(len(sub)), test_size=0.3, random_state=0,
                          stratify=sub["category"])

def run(col, ngram):
    v = TfidfVectorizer(token_pattern=r"\w+", ngram_range=ngram,
                        min_df=3, max_features=40000)
    A = v.fit_transform(sub[col].iloc[tr]); B = v.transform(sub[col].iloc[te])
    m = LogisticRegression(max_iter=1000).fit(A, sub["category"].iloc[tr])
    return accuracy_score(sub["category"].iloc[te], m.predict(B))

rows = []
for col, label in [("text", "original"), ("scrambled", "shuffled")]:
    for ngram, gl in [((1,1), "unigrams"), ((1,2), "uni+bigrams")]:
        rows.append({"text": label, "features": gl, "accuracy": run(col, ngram)})
res = pd.DataFrame(rows)
print(res.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

In [ ]:
orig_uni = res[(res.text=="original") & (res.features=="unigrams")]["accuracy"].iloc[0]
scr_uni  = res[(res.text=="shuffled") & (res.features=="unigrams")]["accuracy"].iloc[0]
orig_bi  = res[(res.text=="original") & (res.features=="uni+bigrams")]["accuracy"].iloc[0]
print(f"order signal (original - shuffled, unigrams): {orig_uni - scr_uni:+.4f}")
print(f"bigram gain  (uni+bi - uni, original)       : {orig_bi - orig_uni:+.4f}")
print()
if abs(orig_uni - scr_uni) < 0.01:
    print("Essentially no order signal. An LSTM or transformer has nothing to")
    print("recover here, and any gain it shows over a bag of words is coming")
    print("from capacity, not from sequence. Do not buy the model for this task.")
else:
    print("There IS order signal. A sequence model could plausibly recover it,")
    print("and the shuffled score is the baseline it must beat by more than noise.")

fig, ax = plt.subplots(figsize=(7, 3.8))
x = np.arange(2)
ax.bar(x-0.2, [orig_uni, orig_bi], width=.4, label="original order")
ax.bar(x+0.2, [scr_uni, res[(res.text=="shuffled") & (res.features=="uni+bigrams")]["accuracy"].iloc[0]],
       width=.4, label="shuffled")
ax.set_xticks(x); ax.set_xticklabels(["unigrams", "uni+bigrams"])
ax.set_ylabel("accuracy"); ax.legend(); ax.set_title("How much does order carry?")
plt.tight_layout()

In [ ]:
# A task where order DOES matter, for contrast: detecting negation.
NEG_PAIRS = [
    ("the policy was approved", "the policy was not approved"),
    ("funding has been released", "funding has not been released"),
    ("the report confirms the finding", "the report does not confirm the finding"),
]
cv = CountVectorizer(token_pattern=r"\w+")
for a, b in NEG_PAIRS:
    va, vb = cv.fit_transform([a, b]).toarray()
    shared = (np.minimum(va, vb).sum()) / max(va.sum(), 1)
    print(f"   {shared:.0%} token overlap: {a!r} vs {b!r}")
print()
print("These pairs mean opposite things and share most of their tokens. Any task")
print("whose labels turn on negation, attribution or agency has real order")
print("signal — and that is where module 24b's LSTM and module 25's transformer")
print("earn their cost.")

## The decision rule

1. **Run the shuffle test.** It costs minutes.
2. **If the gap is under a point, stop.** Your task is a bag-of-words task; use
   module 7 and spend the saved week on the corpus instead.
3. **If the gap is large, check bigrams first.** They capture local order at a
   fraction of the cost, and often close most of the gap.
4. **Only then consider a sequence model**, and hold it to beating the bigram
   baseline, not the unigram one.

**Defensible:** "A shuffled-token control showed an order signal of [X] points;
bigram features recovered [Y] of it, so a sequence model was not pursued."

That is a sentence reviewers rarely see and always respect, because it shows the
choice was made on evidence.

### 🔧 Your turn 1

Construct a small labelled set where the label depends on negation, and run the shuffle test on it. How large is the gap?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

Large, and that is the point — the test is sensitive when there is something to
detect. This makes the near-zero result on topic classification informative
rather than merely negative: it is not that the test cannot find order signal,
it is that topic labels do not contain any. Knowing which of those two
situations you are in is the whole value of running the control.

</details>